In [39]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd



from scipy.stats import permutation_test
sys.path.append(r"G:\apuntes_mne\code_MOUS\scripts_functions_created")

# Ahora importa la función
from print5 import print5

from IPython.display import Markdown, display

import pickle

In [40]:

try:
    # Si se ejecuta como SCRIPT .py: usar __file__
    sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))
except NameError:
    # Si se ejecuta como NOTEBOOK Jupyter: usar path relativo
    sys.path.append("..")  # sube un nivel desde la carpeta actual del notebook


# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"
#subj = sys.argv[1] ## name of participant list


# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")
    
    
    
with open(datadir / f"subjects_remove_{modality}.pkl", "rb") as f:
    subjects_remove = pickle.load(f)


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
output_preproc       → g:\MOUS_204\MOUS_visual\output_preproc
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_pat

# Tables and paths
Now, as specially in the event_preprocessing we are going to work with lots of different comparisons, we are going to create, instead of variables only, we are going to use comparisons 

Example: comparison_1= [zinnen, woorden]
so variable 1= comparison_1[0]
### Paths
ACW_path
PLE_path

### Tables and metrics
#### Block

*ACW*

f"acw_results_subjects_all_{layer_script}.pickle"

f"autocorrelation_subjects_all_{layer_script}.pickle"

**variables** =  acw_50_elect_all_epoch_all //  'intercept_y_elect_all_epoch_all', 'slope_elect_all_epoch_all'


*PLE*
f"table_PLE_slope_intercept_subjects_all_{layer_script}.pickle"

f"table_PLE_subjects_all_{layer_script}.pickle"


#### dynamic

f"dynamic_autocorrelation_results_subjects_all_{layer_script}.pickle"

f"dynamic_autocorrelation_subjects_all_{layer_script}.pickle"

**variables** = acw_50_slope_elect_all_epoch_all   // acw_50_std_elect_all_epoch_all 

conditions = (['zinnen_RC_plus', 'zinnen_RC_neg', 'zinnen_RC_neg_question_hit',
       'woorden_RC_neg', 'zinnen_RC_plus_question_hit',
       'zinnen_RC_neg_question_incorrect',
       'woorden_RC_neg_question_incorrect', 'woorden_RC_neg_question_hit',
       'zinnen_RC_plus_question_incorrect', 'woorden_RC_plus',
       'woorden_RC_plus_question_hit',
       'woorden_RC_plus_question_incorrect']')



In [ ]:

## Variables of script
##path=analysis_path
path = ACW_path
name_table=f"acw_results_subjects_all_{layer_script}.pickle"


table_df = pd.read_pickle(f"{path}\\{name_table}")

# Extraer sujetos del dataframe
subjects = table_df["Subject"].unique()

# Filtrar sujetos eliminados
subjects = [s for s in subjects if s not in subjects_remove]

# Filtrar tabla también
table_df = table_df.query("Subject in @subjects")




# Remove them also from table
#METRIC NAMES
metric='acw_50_elect_all_epoch_all'


if '_elect_all_epoch_all' in metric:
    print("yes")
    metric_name = metric.replace('_elect_all_epoch_all', "")
else:
    metric_name = metric


name_table_fooof =f"df_fooof_subject_all_{layer_script}.pickle"


table_fooof_df = pd.read_pickle(f"{path}\\{name_table_fooof}")



#FOOOF table
# Filtrar tabla también
table_fooof_df = table_fooof_df.query("subject in @subjects")
freq_bands = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 12),
    'beta': (12, 30),
    'gamma': (30, 40)
}  





##CONDITION NAMES
if layer_script == "event":
    zinnen_noQ = ["zinnen_RC_plus", "zinnen_RC_neg"]
    woorden_noQ = ["woorden_RC_plus", "woorden_RC_neg"]

    zinnen_RCplus = ["zinnen_RC_plus"]
    zinnen_RCneg = ["zinnen_RC_neg"]

    question_hit = [
        "zinnen_RC_neg_question_hit",
        "zinnen_RC_plus_question_hit",
        "woorden_RC_neg_question_hit",
        "woorden_RC_plus_question_hit"
    ]

    question_incorrect = [
        "zinnen_RC_neg_question_incorrect",
        "zinnen_RC_plus_question_incorrect",
        "woorden_RC_neg_question_incorrect",
        "woorden_RC_plus_question_incorrect"
    ]



if layer_script == "block":
    ## Comparisons
    comparison_1 = ["ZINNEN", "WOORDEN"]
    comparison_2=[]
    comparison_3=[]
    comparison_4=[]

    comparisons = [comparison_1, comparison_2,
                comparison_3, comparison_4]
elif layer_script == "event":
    comparison_1 = [zinnen_noQ, woorden_noQ]          # ZINNEN vs WOORDEN sin preguntas
    comparison_2 = [zinnen_RCplus, zinnen_RCneg]      # RC+ vs RC− en zinnen
    comparison_3 = [question_hit, question_incorrect] # Trials correctos vs incorrectos

    comparisons = [comparison_1, comparison_2, comparison_3]

#TYPE OF DIFFERENCES
difference = "normal"  # "normal" or "inverse"

## this will be used in permutation differences
if difference == "normal":
    alternative = "greater" 
    # this is for cluster permutation test
    tail=1 
    threshold_direction =1
else:
    alternative="less"
    tail=-1
    threshold_direction =-1

number_decimals=6


# channels = pd.read_csv(channels_structure_path / f"channels_mag_{modality}.csv")
# channels_mag=channels[channels[f"canal_efectivo_{modality}"].notna()][f"canal_efectivo_{modality}"]

# channels_mag=channels_mag.tolist()
# print5(channels_mag)
# indice_channels_efectivos = channels[channels[f"canal_efectivo_{modality}"].notna()]["indice"]
# indice_channels_efectivos=indice_channels_efectivos.tolist()
# del channels

##valores de las columnas
condition = table_df["Condition"].unique()
print5("Condiciones en los datos:", condition)

subjects = table_df["Subject"].unique()
print5("Sujetos en los datos:", subjects)

elect_all=  table_df["Elect"].unique()
print5("sensores en los datos:", elect_all)

epochs_all=  table_df["Epoch"].unique()
print5("Epochs en los datos:", epochs_all)
## get info 

#read epochs to build evoked
# leer epochs del sujeto (único archivo)
epochs = mne.read_epochs(epochs_clean_path / f"{subj}_epochs_{layer_script}-epo.fif")

if layer_script == "block":
# seleccionar ensayos de condición "zinnen"
    epochs_zinnen = epochs["fix_ZINNEN"]
if layer_script == "event":
    # seleccionar ensayos de condición "zinnen"
    epochs_zinnen = epochs["begin_zinnen_RC_neg"]

# obtener evoked
evoked_zinnen = epochs_zinnen.copy().pick("mag", exclude="bads").average()

info = evoked_zinnen.info

# limpieza memoria (opcional)
del epochs, epochs_zinnen



yes
Condiciones en los datos: ['ZINNEN' 'WOORDEN']
Sujetos en los datos: ['sub-V1001' 'sub-V1003' 'sub-V1004' 'sub-V1005' 'sub-V1007' 'sub-V1008'
 'sub-V1009' 'sub-V1011' 'sub-V1012' 'sub-V1013' 'sub-V1015' 'sub-V1016'
 'sub-V1019' 'sub-V1020' 'sub-V1022' 'sub-V1024' 'sub-V1025' 'sub-V1027'
 'sub-V1028' 'sub-V1029' 'sub-V1030' 'sub-V1031' 'sub-V1032' 'sub-V1033'
 'sub-V1034' 'sub-V1035' 'sub-V1036' 'sub-V1037' 'sub-V1038' 'sub-V1039'
 'sub-V1040' 'sub-V1042' 'sub-V1044' 'sub-V1045' 'sub-V1046' 'sub-V1048'
 'sub-V1049' 'sub-V1050' 'sub-V1052' 'sub-V1053' 'sub-V1054' 'sub-V1055'
 'sub-V1057' 'sub-V1058' 'sub-V1059' 'sub-V1061' 'sub-V1062' 'sub-V1063'
 'sub-V1065' 'sub-V1066' 'sub-V1068' 'sub-V1069' 'sub-V1070' 'sub-V1071'
 'sub-V1072' 'sub-V1073' 'sub-V1074' 'sub-V1075' 'sub-V1076' 'sub-V1077'
 'sub-V1079' 'sub-V1080' 'sub-V1081' 'sub-V1083' 'sub-V1084' 'sub-V1085'
 'sub-V1086' 'sub-V1087' 'sub-V1088' 'sub-V1089' 'sub-V1090' 'sub-V1092'
 'sub-V1093' 'sub-V1094' 'sub-V1095' 'sub-V1097' 's

In [42]:
dict_isc= pd.read_pickle(ISC_block_path /f"ISC_results_block.pkl")
dict_woorden_block=dict_isc['dict_isc_WOORDEN']
dict_zinnen_block = dict_isc['dict_isc_ZINNEN']
##channels significant adjusted in each condition
numbers_channels_woorden= dict_woorden_block["significant_channels_adjusted"]
numbers_channels_zinnen= dict_zinnen_block["significant_channels_adjusted"]
print("numbers_channels_woorden", numbers_channels_woorden, "numbers_channels_zinnen", numbers_channels_zinnen)
print("len(numbers_channels_woorden)",len(numbers_channels_woorden),  "len(numbers_channels_zinnen)",len(numbers_channels_zinnen))

names_channels_woorden=dict_woorden_block["significant_channels_adjusted_names"]
names_channels_zinnen=dict_zinnen_block["significant_channels_adjusted_names"]

print("names_channels_woorden", names_channels_woorden)
print("names_channels_zinnen", names_channels_zinnen)

h_subj =0
path_epochs = epochs_clean_path / f"{subjects[h_subj]}_epochs_{layer_script}-epo.fif"
epochs = mne.read_epochs(path_epochs, preload=False)
#establecimiento de canales palabras, canales frases y canales mixtos
# Tomamos el orden original de los canales del objeto epochs
all_channels = epochs.ch_names
all_channels_numbers= [epochs.ch_names.index(ch) for ch in epochs.ch_names]


del epochs

# Palabras
numbers_channels_only_woorden = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch not in numbers_channels_zinnen
]

numbers_channels_only_zinnen = [
    ch for ch in all_channels_numbers if ch in numbers_channels_zinnen and ch not in numbers_channels_woorden
]

numbers_channels_intersection = [
    ch for ch in all_channels_numbers if ch in numbers_channels_woorden and ch in numbers_channels_zinnen
]

# Lo mismo pero usando nombres
names_channels_only_woorden = [
    ch for ch in all_channels if ch in names_channels_woorden and ch not in names_channels_zinnen
]

names_channels_only_zinnen = [
    ch for ch in all_channels if ch in names_channels_zinnen and ch not in names_channels_woorden
]

names_channels_intersection = [
    ch for ch in all_channels if ch in names_channels_woorden and ch in names_channels_zinnen
]
# ---------------------------
# Print resumen
# ---------------------------

print(f"len(significant_channels_only_woorden): {len(numbers_channels_only_woorden)}, "
      f"len(significant_channels_only_zinnen): {len(numbers_channels_only_zinnen)}, "
      f"len(significant_channels_intersection): {len(numbers_channels_intersection)}")



print("\n✅ Only woorden (names):", names_channels_only_woorden)
print("✅ Only zinnen (names):", names_channels_only_zinnen)
print("✅ Intersection (names):", names_channels_intersection)

numbers_channels_woorden [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  17  18  19
  21  31  35  36  40  41  42  43  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  60  61  62  63  64  65  66  68  69  70  72  73  75  77
  78  79  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95
  96  97  98  99 100 101 102 103 104 105 106 107 108 110 111 112 113 114
 115 116 117 118 119 120 121 122 123 124 125 126 127 128 131 132 133 134
 135 136 137 138 139 140 141 142 143 144 146 147 148 149 150 151 161 166
 172 177 178 182 183 184 188 192 195 196 198 199 200 201 202 204 206 207
 209 210 211 212 214 215 216 217 221 222 223 224 225 226 227 229 230 231
 232 233 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251
 252 253 254 255 256 257 258] numbers_channels_zinnen [  2   3   4   5   7   8   9  10  11  12  13  14  17  18  19  21  23  24
  25  26  28  29  30  31  32  33  34  35  36  37  38  39  40  41  42  43
  44  45  46  47  48  49  50  51  52  53  54 

## Posibles comparaciones estadísticas
len(significant_channels_only_woorden): 25, len(significant_channels_only_zinnen): 92, len(significant_channels_intersection) 88

Primer problema, el numero de canales zinnen es claramente superior al de woorden, big problem, aunque la interseccion es alta, tal como esperaríamos. Seguramente esto sea porque hay ruido 

- comparación ACW-50 en zinnen entre estos canales zinnen y canales woorden  + comapracion en woorden de lo mismo

- comparación entre solo zinnen con intersection  en woorden y en zinnen (aunque ojo, esto es básicamente aceptar que los canales solo Zinnen son ruido)

- comapración en el promedio general de acw en ambas condiciones de los canalaes zinnen vs promedio general de canales woorden

- comparación de la intersección vs canales no signficativos: no sé muy bien como interpretar esto

- 

In [43]:
#filtras por sujeto 
channels_type=["ch_only_woorden", "ch_only_zinnen", "ch_intersection"]


dict_metric_condition_subj_epoch_mean_ch_type = {}
dict_metric_condition_subj_epoch_mean_ch_type_mean = {}

for comparison in comparisons:
    if comparison:
            
        for exp_condition in comparison:
            
            for ch_type in channels_type:
                # for subj in subjects:
                    #creacion de lista de valores de metric para cada sujeto
                    
                #eliges la condicion y el tipo de canal y te quedas con los valores 
                if ch_type == "ch_only_woorden":
                    elects = names_channels_only_woorden
                elif ch_type == "ch_only_zinnen":
                    elects = names_channels_only_zinnen
                elif ch_type == "ch_intersection":
                    elects = names_channels_intersection
                
                print(f"ch_type is {ch_type}, exp_condition is {exp_condition}, elects is  {elects}")
                array_metric_condition_subj_epoch_mean_ch_type = (
                    table_df
                    .query("Elect in @elects")
                    .query("Condition == @exp_condition")
                    .groupby(["Subject", "Elect"])[metric]
                    .mean().reset_index()
                    .pivot(index="Subject", columns="Elect", values=metric)
                    .to_numpy()
                )
                
                array_metric_condition_subj_epoch_mean_ch_type_mean = (
                    table_df
                    .query("Elect in @elects")
                    .query("Condition == @exp_condition")
                    .groupby("Subject")[metric]
                    .mean()
                    .sort_index()    
                    .values
                )
                
                        
                
                dict_metric_condition_subj_epoch_mean_ch_type[f"{exp_condition}_{ch_type}"] = array_metric_condition_subj_epoch_mean_ch_type
                dict_metric_condition_subj_epoch_mean_ch_type_mean[f"{exp_condition}_{ch_type}"] = array_metric_condition_subj_epoch_mean_ch_type_mean


ch_type is ch_only_woorden, exp_condition is ZINNEN, elects is  ['MLC12-4304', 'MLC13-4304', 'MLC21-4304', 'MLO31-4304', 'MLO32-4304', 'MLO33-4304', 'MLO42-4304', 'MLO43-4304', 'MRF55-4304', 'MRF64-4304']
ch_type is ch_only_zinnen, exp_condition is ZINNEN, elects is  ['MLF11-4304', 'MLF12-4304', 'MLF13-4304', 'MLF14-4304', 'MLF22-4304', 'MLF23-4304', 'MLF24-4304', 'MLF31-4304', 'MLF32-4304', 'MLF33-4304', 'MLF41-4304', 'MLF42-4304', 'MLF43-4304', 'MLF52-4304', 'MLO21-4304', 'MLT31-4304', 'MRC11-4304', 'MRC12-4304', 'MRC51-4304', 'MRF11-4304', 'MRF12-4304', 'MRF13-4304', 'MRF14-4304', 'MRF21-4304', 'MRF22-4304', 'MRF23-4304', 'MRF24-4304', 'MRF31-4304', 'MRF32-4304', 'MRF33-4304', 'MRF34-4304', 'MRF41-4304', 'MRF42-4304', 'MRF43-4304', 'MRF44-4304', 'MRF51-4304', 'MRF52-4304', 'MRF61-4304', 'MRO11-4304', 'MRO13-4304', 'MRO21-4304', 'MRO22-4304', 'MRO23-4304', 'MRO31-4304', 'MRO32-4304', 'MRP41-4304', 'MRP51-4304', 'MRP53-4304', 'MRT14-4304', 'MRT24-4304', 'MRT25-4304', 'MZF01-4304', 'MZ

In [44]:
## rename columns FOOOF table if necessary

table_fooof_df = table_fooof_df.rename(columns={
    "subject": "Subject",
    "condition": "Condition",
    "epoch": "Epoch",
    "channel": "Elect"
})



In [45]:
## this is applied only now because i have index for channels, not names

sensor_map = {sensor: i for i, sensor in enumerate(elect_all)}
table_df['Elect'] = table_df['Elect'].map(sensor_map)
#merge tables
table_df["Condition"] = table_df["Condition"].str.lower()

table_merged_df = pd.merge(table_df, table_fooof_df, on=["Subject", "Condition", "Epoch", "Elect"])

In [46]:
metrics = [metric] + list(freq_bands.keys())
metrics

['acw_50_elect_all_epoch_all', 'delta', 'theta', 'alpha', 'beta', 'gamma']

In [47]:
## MODIFY THIS TO USE NAMES INSTEAD OF NUMBERS

# create a new column named ch type, based on the channel names
conditions = [
    table_merged_df["Elect"].isin(numbers_channels_only_zinnen),
    table_merged_df["Elect"].isin(numbers_channels_only_woorden),
    table_merged_df["Elect"].isin(numbers_channels_intersection)
]

choices = [
    "ch_only_zinnen",
    "ch_only_woorden",
    "ch_intersection"
]

table_merged_df["Elect_type"] = np.select(conditions, choices, default="unknown")


In [48]:
table_merged_df_filtered = table_merged_df[
    table_merged_df["Elect_type"].isin(["ch_intersection", "ch_only_zinnen"])
]

In [49]:
metrics_with_fooof = [metric] + list(freq_bands.keys())

# this is for comparing between conditions, as it has dimension=number of subjects
table_df_merged_cond_subj_epoch_mean_elect_mean= table_merged_df_filtered.groupby(["Subject", "Condition", "Elect_type"])[metrics_with_fooof].mean().reset_index()

#this is for plotting results (plot_topomap) as it has dimension=number of electrodes
table_df_merged_cond_subj_mean_epoch_mean_elect=table_merged_df_filtered.groupby(["Condition", "Elect"])[metrics_with_fooof].mean().reset_index()

#this is for cluster permutation test, as it has dimension=number of subjects and electrodes 
table_df_merged__cond_subj_epoch_mean_elect=table_merged_df_filtered.groupby(["Subject", "Condition", "Elect"])[metrics_with_fooof].mean().reset_index()

# 1. Comparaciones ch_Zinnen only vs ch_intersection
These are gonne be comparisons between channels zinnen vs intersection in each condition.
E.g. in zinnen condition, we compare acw in 


In [50]:

type_channel=["intersection", "only_woorden"]
# type_channel=[ "woorden"]

## 1.1 Comparisons in each experimental condition

In [51]:
import statsmodels.formula.api as smf


if difference == "normal":     
    display(Markdown("**Difference is normal**, alternative hypothesis is **zinnen > woorden**")) 
else:     
    display(Markdown("**Difference is inverse**, alternative hypothesis is **zinnen < woorden** (for PLE condition)"))


# # if difference == "normal":
# #     print(f"Difference is normal, alternative hypothesis is zinnen > woorden")
# # else:
# #     print(f"Difference is inverse, alternative hypothesis is less (for PLE condition) alternative hypothesis is zinnen < woorden")

# dependency= ["Dependent", "Independent"]
# # for type_dependency in dependency:
#     print("------------------------------------------------------------------------------")


#     print(f"{type_dependency.upper()} analysis in {metric_name}")
        
for comparison in comparisons:
    
    if comparison:
        print("------------------------------------------------------------------------------")
        print(f"Comparison is between: {comparison}")
        for experimental_condition in comparison:
            print(f"\nExperimental condition: {experimental_condition}")
            print("------------------------------------------------------------------------------")
            
            # Filtrar datos por condición experimental
            data = table_df_merged_cond_subj_epoch_mean_elect_mean[
                table_df_merged_cond_subj_epoch_mean_elect_mean["Condition"] 
                == experimental_condition.lower()
            ]

            # ==========================================================
            # 1️⃣ MODELO MIXTO SIMPLE (solo Elect_type)
            # ==========================================================
            print("\n----- Mixed model (Elect_type only) -----")
            print("------------------------------------------------------------------------------")

            formula = f"{metric} ~ Elect_type"

            modelo = smf.mixedlm(
                formula,
                data=data,
                groups="Subject"
            ).fit()

            print(modelo.summary())

            # ==========================================================
            # 2️⃣ MODELO MIXTO CON PARÁMETROS FOOOF
            # ==========================================================
            print("\n----- Mixed model WITH FOOOF parameters -----")
            print("------------------------------------------------------------------------------")

            formula = f"{metric} ~ Elect_type + " + " + ".join(list(freq_bands.keys()))

            modelo = smf.mixedlm(
                formula,
                data=data,
                groups="Subject"
            ).fit()

            print(modelo.summary())
            
            
 

**Difference is normal**, alternative hypothesis is **zinnen > woorden**

------------------------------------------------------------------------------
Comparison is between: ['ZINNEN', 'WOORDEN']

Experimental condition: ZINNEN
------------------------------------------------------------------------------

----- Mixed model (Elect_type only) -----
------------------------------------------------------------------------------


c:\Users\UCM\anaconda3\envs\env_meg\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                 Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: acw_50_elect_all_epoch_all
No. Observations: 190     Method:             REML                      
No. Groups:       95      Scale:              0.0000                    
Min. group size:  2       Log-Likelihood:     784.0680                  
Max. group size:  2       Converged:          Yes                       
Mean group size:  2.0                                                   
------------------------------------------------------------------------
                              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------
Intercept                      0.020    0.000 45.941 0.000  0.020  0.021
Elect_type[T.ch_only_zinnen]  -0.001    0.000 -2.173 0.030 -0.001 -0.000
Subject Var                    0.000    0.001                           


----- Mixed model WITH FOOOF parameters -----
---------------------

c:\Users\UCM\anaconda3\envs\env_meg\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                 Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: acw_50_elect_all_epoch_all
No. Observations: 190     Method:             REML                      
No. Groups:       95      Scale:              0.0000                    
Min. group size:  2       Log-Likelihood:     775.2064                  
Max. group size:  2       Converged:          Yes                       
Mean group size:  2.0                                                   
------------------------------------------------------------------------
                              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------
Intercept                      0.030    0.002 12.636 0.000  0.026  0.035
Elect_type[T.ch_only_zinnen]  -0.002    0.000 -4.172 0.000 -0.003 -0.001
delta                          0.027    0.017  1.604 0.109 -0.006  0.061
theta                         -0.000    0.002 -0.171 0.864 -0.005  0.

c:\Users\UCM\anaconda3\envs\env_meg\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                 Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: acw_50_elect_all_epoch_all
No. Observations: 190     Method:             REML                      
No. Groups:       95      Scale:              0.0000                    
Min. group size:  2       Log-Likelihood:     803.7617                  
Max. group size:  2       Converged:          Yes                       
Mean group size:  2.0                                                   
------------------------------------------------------------------------
                              Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------------------
Intercept                      0.020    0.000 48.550 0.000  0.019  0.021
Elect_type[T.ch_only_zinnen]  -0.001    0.000 -3.238 0.001 -0.002 -0.000
Subject Var                    0.000    0.001                           


----- Mixed model WITH FOOOF parameters -----
---------------------

c:\Users\UCM\anaconda3\envs\env_meg\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 1.2 Using average in both conditions


In [52]:
# if difference == "normal":
#     display(Markdown("**Difference is normal**, alternative hypothesis is **zinnen > woorden**"))
# else:
#     display(Markdown("**Difference is inverse**, alternative hypothesis is **zinnen < woorden** (for PLE condition)"))

# # # if difference == "normal":
# # #     print(f"Difference is normal, alternative hypothesis is zinnen > woorden")
# # # else:
# # #     print(f"Difference is inverse, alternative hypothesis is less (for PLE condition) alternative hypothesis is zinnen < woorden")

# dependency = ["Dependent", "Independent"]

# for type_dependency in dependency:
#     print("------------------------------------------------------------------------------")
#     print("------------------------------------------------------------------------------")
#     print(f"{type_dependency.upper()} analysis in {metric_name}")

#     for ch in type_channel:
#         print("------------------------------------------------------------------------------")
#         print(f"for differences between only_zinnen and {ch}:")
#         print("------------------------------------------------------------------------------")
#         # Promediar condiciones para X y Y
#         x = (
#             dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[0]}_ch_only_zinnen"]
#             + dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[1]}_ch_only_zinnen"]
#         ) / 2

#         y = (
#             dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[0]}_ch_{ch}"]
#             + dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[1]}_ch_{ch}"]
#         ) / 2

#         # Calcular diferencia
#         diff = x - y

#         if type_dependency == "Dependent":
#             res = permutation_test(
#                 (diff, np.zeros_like(diff)),
#                 statistic=paired_statistic,
#                 vectorized=False,
#                 n_resamples=10000,
#                 alternative=alternative,
#                 random_state=42
#             )

#             # Cohen's d (dependiente)
#             mean_diff = np.mean(diff)
#             std_diff = np.std(diff, ddof=1)
#             cohens_d = mean_diff / std_diff

#         elif type_dependency == "Independent":
#             res = permutation_test(
#                 (x, y),
#                 statistic=diff_means,
#                 vectorized=False,
#                 n_resamples=10000,
#                 alternative=alternative,
#                 random_state=42
#             )

#             # Cohen's d (independiente)
#             mean_x, mean_y = np.mean(x), np.mean(y)
#             std_x, std_y = np.std(x, ddof=1), np.std(y, ddof=1)
#             n_x, n_y = len(x), len(y)

#             pooled_std = np.sqrt(
#                 ((n_x - 1) * std_x**2 + (n_y - 1) * std_y**2) / (n_x + n_y - 2)
#             )

#             if difference == "normal":
#                 cohens_d = (mean_x - mean_y) / pooled_std
#             else:
#                 cohens_d = (mean_y - mean_x) / pooled_std

#         # Imprimir resultados
#         print(f"   Statistic value: {res.statistic:.{number_decimals}f}")
#         print(f"   p-value        : {res.pvalue:.{number_decimals}f}")
#         print(f"   Cohen's d      : {cohens_d:.{number_decimals}f}")

# 2. Comparison of each channel type in both conditions

Before we compared differences of word and sentence channels. Now we are going to compare the channels with themselves in different experimental conditions

In [53]:


# if difference == "normal":
#     display(Markdown("**Difference is normal**, alternative hypothesis is **zinnen > woorden**"))
# else:
#     display(Markdown("**Difference is inverse**, alternative hypothesis is **zinnen < woorden** (for PLE condition)"))


# # here i have to take the list of ALL types of channels, not like before, where we always take only_zinnen
# all_type_channels = ['only_zinnen','intersection', 'only_woorden']

# for ch in all_type_channels:
#     print("------------------------------------------------------------------------------")
#     print(f"for differences in {metric_name} ch {ch}  between zinnen and word condition:")
#     # print(f"Processing {experimental_condition} for differences between zinenn and {ch}...")

#     #ch value in zinnen condition
#     x= dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[0]}_ch_{ch}"]
#     y= dict_metric_condition_subj_epoch_mean_ch_type_mean[f"X_{experimental_condition[1]}_ch_{ch}"]

#     print(f"X is: X_{experimental_condition[0]}_ch_{ch}", x.shape)
#     print(f"Y is: X_{experimental_condition[1]}_ch_{ch}", y.shape)


#     if difference=="normal":
#         diff=x-y
#     elif difference=="inverse":
#         diff=y -x
#         # Permutation test
#     res = permutation_test(
#         (diff, np.zeros_like(diff)),
#         statistic=paired_statistic,
#         vectorized=False,
#         n_resamples=10000,
#         alternative=alternative, 
#         random_state=42
#     )

#     # Calcular Cohen's d para muestras emparejadas
#     mean_diff = np.mean(diff)
#     std_diff = np.std(diff, ddof=1)
    
#     if difference == "normal":
#         cohens_d = (mean_x - mean_y) / pooled_std
#     else:
#         cohens_d = (mean_y - mean_x) / pooled_std
        
#     # Imprimir resultados
#     print(f"   Statistic value: {res.statistic:.{number_decimals}f}")
#     print(f"   p-value        : {res.pvalue:.{number_decimals}f}")
#     print(f"   Cohen's d      : {cohens_d:.{number_decimals}f}")